# From Raw Text to Structured Inputs

## MSA 8700 — Module 8: NLP and Text Processing

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Describe the **preprocessing pipeline** that turns messy, real-world text into structured data
2. Write and interpret **regular expressions** for deterministic pattern matching
3. Use **Beautiful Soup** to parse HTML and extract structured fields from web pages
4. Understand how these deterministic techniques fit into **agentic AI workflows** alongside LLMs

---

# Part 1: Why Preprocessing Matters

## The Problem: Messy Real-World Text

Agentic AI systems rarely receive perfectly formatted input. In practice, they start from **messy text** that comes from a wide variety of sources:

| Source | What It Looks Like | Challenges |
|--------|-------------------|------------|
| **Emails** | Free-form text with headers, signatures, forwarded chains | Mixed structure, varying formats |
| **Web pages** | HTML with navigation, ads, scripts, and content | Need to separate signal from noise |
| **Log files** | Timestamped entries with error codes and stack traces | Semi-structured, high volume |
| **PDFs** | Extracted text with broken formatting, tables as text | Layout information lost |
| **Chat transcripts** | Multi-turn conversations with timestamps and usernames | Context spread across messages |

Before an LLM can **reason**, **act**, or **call tools**, you often need **deterministic preprocessing** — reliable, rule-based transformations that produce consistent results every time.

## The Preprocessing Pipeline

A typical text preprocessing pipeline follows four stages:

```
Step 1: Collect text from heterogeneous sources (web pages, APIs, databases)
            ↓
Step 2: Normalize and clean it (strip HTML, handle encoding, remove boilerplate)
            ↓
Step 3: Extract structured elements (titles, dates, entities, amounts)
            ↓
Step 4: Feed cleaned text and structured fields into LLM tools,
        retrieval systems, or classical models
```

### Why not just send everything to the LLM?

You might wonder: if LLMs are so capable, why not skip preprocessing and send raw HTML or messy text directly?

There are several important reasons:

- **Cost**: LLM API calls are priced per token. Sending raw HTML with navigation bars, scripts, and ads wastes tokens (and money) on irrelevant content.
- **Accuracy**: For structured fields like dates, invoice numbers, and prices, regex is **100% reliable** on known formats. LLMs can hallucinate or misparse these.
- **Speed**: Regex and HTML parsing run in microseconds. LLM calls take seconds.
- **Determinism**: The same regex on the same input always gives the same output. LLMs are stochastic — they may give slightly different answers each time.
- **Context window limits**: Sending huge documents may exceed the LLM's context window or degrade its performance on long inputs.

---

# Part 2: Regular Expressions

## What Are Regular Expressions?

Regular expressions (regex) provide **deterministic pattern matching** for semi-structured text. They let you describe a pattern — like "four digits, a dash, two digits, a dash, two digits" — and find every occurrence in a body of text.

Key properties of regex:

- **Fast**: Compiled patterns run in microseconds, even on large texts
- **Transparent**: The pattern is the logic — easy to audit, test, and debug
- **Embeddable**: Easy to include in rule-based components of agent workflows
- **Deterministic**: Same input + same pattern = same output, every time

## Regex Syntax Quick Reference

Before diving into examples, here is a quick reference of the most common regex building blocks:

| Pattern | Meaning | Example Match |
|---------|---------|---------------|
| `.` | Any single character | `a.c` matches `abc`, `a1c`, `a-c` |
| `\d` | Any digit (0-9) | `\d\d` matches `42`, `07` |
| `\w` | Any word character (letter, digit, underscore) | `\w+` matches `hello`, `var_1` |
| `\s` | Any whitespace (space, tab, newline) | `\s+` matches one or more spaces |
| `*` | Zero or more of the preceding | `ab*c` matches `ac`, `abc`, `abbc` |
| `+` | One or more of the preceding | `ab+c` matches `abc`, `abbc` but not `ac` |
| `?` | Zero or one of the preceding | `colou?r` matches `color` and `colour` |
| `{n}` | Exactly n of the preceding | `\d{4}` matches `2026` |
| `{n,m}` | Between n and m of the preceding | `\d{2,4}` matches `42`, `123`, `2026` |
| `[]` | Character class (any one of) | `[aeiou]` matches any vowel |
| `()` | Capture group | `(\d{4})-(\d{2})` captures year and month |
| `\|` | Alternation (or) | `cat\|dog` matches `cat` or `dog` |
| `^` | Start of string | `^Hello` matches `Hello world` |
| `$` | End of string | `world$` matches `Hello world` |

## Example 1: Extracting Invoice Numbers and Dates from Email

Imagine a **finance agent** that processes incoming emails. Before asking an LLM to reason about the email ("Is this invoice overdue?"), we first use regex to reliably extract the structured fields.

In [ ]:
import re

text = """
Dear Peter,
Your invoice INV-2026-031 was issued on 2026-03-01.
Please pay by 2026-03-15.
"""

# Pattern: "INV-" followed by exactly 4 digits, a dash, then exactly 3 digits
invoice_pattern = r"INV-\d{4}-\d{3}"

# Pattern: exactly 4 digits, dash, 2 digits, dash, 2 digits (ISO date format)
date_pattern = r"\d{4}-\d{2}-\d{2}"

invoice_ids = re.findall(invoice_pattern, text)
dates = re.findall(date_pattern, text)

print("Invoice IDs found:", invoice_ids)
print("Dates found:", dates)

### Breaking Down the Patterns

Let's trace through each pattern character by character:

**Invoice pattern** `INV-\d{4}-\d{3}`:
- `INV-` — matches the literal text "INV-"
- `\d{4}` — matches exactly 4 digits (e.g., `2026`)
- `-` — matches a literal dash
- `\d{3}` — matches exactly 3 digits (e.g., `031`)

**Date pattern** `\d{4}-\d{2}-\d{2}`:
- `\d{4}` — matches exactly 4 digits for the year
- `-` — literal dash
- `\d{2}` — 2 digits for the month
- `-` — literal dash
- `\d{2}` — 2 digits for the day

> **`re.findall()`** returns a list of all non-overlapping matches in the string. If there are no matches, it returns an empty list.

## Example 2: Extracting Email Addresses

A common task for any agent that processes communication data. Let's build up the pattern step by step.

In [ ]:
message = """
Please contact support@example.com for billing questions,
or reach out to peter.molnar@university.edu for academic inquiries.
Our spam filter caught a message from bot123@suspicious-site.net.
"""

# Email pattern breakdown:
#   [\w.-]+    one or more word chars, dots, or dashes (local part)
#   @          literal @ symbol
#   [\w.-]+    one or more word chars, dots, or dashes (domain)
email_pattern = r"[\w.-]+@[\w.-]+"

emails = re.findall(email_pattern, message)
print("Email addresses found:")
for email in emails:
    print(f"  - {email}")

## Example 3: Extracting Dollar Amounts

Financial agents frequently need to pull dollar amounts from text. Note how we use **capture groups** `()` to extract just the numeric part.

In [ ]:
expense_report = """
Travel expenses for March 2026:
  - Flight: $1,250.00
  - Hotel (3 nights): $687.50
  - Meals: $142.75
  - Ground transport: $89.00
Total: $2,169.25
"""

# Pattern breakdown:
#   \$          literal dollar sign (escaped because $ means end-of-string in regex)
#   [\d,]+      one or more digits or commas (handles "1,250")
#   \.          literal decimal point
#   \d{2}       exactly two decimal digits
amount_pattern = r"\$([\d,]+\.\d{2})"

amounts = re.findall(amount_pattern, expense_report)
print("Dollar amounts found:")
for amount in amounts:
    # Convert to float by removing commas
    value = float(amount.replace(",", ""))
    print(f"  ${amount} -> {value}")

# Calculate total from extracted amounts
total = sum(float(a.replace(",", "")) for a in amounts[:-1])  # exclude the "Total" line
print(f"\nSum of line items: ${total:,.2f}")
print(f"Stated total:      ${amounts[-1]}")

### Key Concept: Capture Groups

Notice that the pattern uses parentheses: `\$([\d,]+\.\d{2})`

- **Without** parentheses, `re.findall()` returns the entire match (e.g., `$1,250.00`)
- **With** parentheses, `re.findall()` returns only the captured group (e.g., `1,250.00`)

This is useful when you need to match a broader context but extract only a specific piece.

## Example 4: Using `re.search()` vs `re.findall()` vs `re.sub()`

Python's `re` module provides several functions. Here's when to use each:

| Function | Purpose | Returns |
|----------|---------|--------|
| `re.findall(pattern, text)` | Find **all** matches | List of strings |
| `re.search(pattern, text)` | Find **first** match | Match object (or None) |
| `re.sub(pattern, repl, text)` | **Replace** matches | New string |
| `re.match(pattern, text)` | Match at **start** of string only | Match object (or None) |

In [ ]:
log_entry = "[2026-03-01 14:23:07] ERROR: Connection to database timed out after 30s"

# re.search() — find the first match and extract groups
timestamp_pattern = r"\[(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2})\]"
match = re.search(timestamp_pattern, log_entry)

if match:
    print(f"Full match:  {match.group(0)}")
    print(f"Date (group 1): {match.group(1)}")
    print(f"Time (group 2): {match.group(2)}")

In [ ]:
# re.sub() — replace matches with new text
# Example: Redact phone numbers in customer data before sending to an LLM
customer_note = "Call customer at 555-867-5309 or 555-123-4567 to follow up on complaint."

phone_pattern = r"\d{3}-\d{3}-\d{4}"
redacted = re.sub(phone_pattern, "[REDACTED]", customer_note)

print("Original: ", customer_note)
print("Redacted: ", redacted)

> **Agentic use case**: Before sending customer data to an LLM for analysis, a preprocessing step uses `re.sub()` to **redact** personally identifiable information (PII) like phone numbers, social security numbers, and credit card numbers. This is a critical safety measure.

## Example 5: Parsing Structured Log Files

Agents that monitor systems need to parse log files efficiently. Here we extract structured records from semi-structured text using **named capture groups** `(?P<name>...)`.

In [ ]:
log_data = """
[2026-03-01 10:15:32] INFO: User alice logged in from 192.168.1.42
[2026-03-01 10:15:45] WARNING: Disk usage at 87% on /dev/sda1
[2026-03-01 10:16:01] ERROR: Failed to write to /var/log/app.log - Permission denied
[2026-03-01 10:16:15] INFO: User bob logged in from 10.0.0.7
[2026-03-01 10:17:03] ERROR: Database connection pool exhausted (max=50)
"""

# Named capture groups make the code more readable
log_pattern = r"\[(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2})\] (?P<level>\w+): (?P<message>.+)"

# re.finditer() returns match objects (unlike findall which returns strings)
for match in re.finditer(log_pattern, log_data):
    print(f"{match.group('level'):8s} | {match.group('date')} {match.group('time')} | {match.group('message')}")

In [ ]:
# Filter for just ERROR entries — a common preprocessing step before
# sending error context to an LLM for root-cause analysis
errors = [match for match in re.finditer(log_pattern, log_data)
          if match.group('level') == 'ERROR']

print(f"Found {len(errors)} error(s):")
for err in errors:
    print(f"  [{err.group('time')}] {err.group('message')}")

---

# Part 3: HTML and Structured Parsing with Beautiful Soup

## Why Parse HTML?

For **web-facing agents** — research agents, monitoring agents, lead generation agents — HTML is one of the most common input formats. But raw HTML is full of noise:

```html
<nav>... navigation links ...</nav>
<div class="sidebar-ad">... advertisement ...</div>
<script>... JavaScript code ...</script>
<article>... this is what we actually want ...</article>
```

**Beautiful Soup** is a Python library that makes it easy to:

- Navigate and search HTML/XML documents by tags, classes, and attributes
- Extract specific text content from targeted elements
- Handle malformed HTML gracefully (real-world HTML is often broken)
- Drop unwanted elements (navigation, ads, scripts)

In [ ]:
# Install Beautiful Soup if not already available
# pip install beautifulsoup4
from bs4 import BeautifulSoup

## Example 1: Extracting Product Information

Imagine a **product research agent** that scrapes e-commerce pages. The structured fields (title, price, description) should be extracted deterministically — not inferred by an LLM.

In [ ]:
html = """
<html>
  <body>
    <h1 class="title">Product ABC</h1>
    <span class="price">$19.99</span>
    <div class="description">This is a durable widget for business use.</div>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")

# .find() locates the first matching element
# .get_text(strip=True) extracts the text content, stripping whitespace
title = soup.find("h1", class_="title").get_text(strip=True)
price = soup.find("span", class_="price").get_text(strip=True)
description = soup.find("div", class_="description").get_text(strip=True)

print(f"Title:       {title}")
print(f"Price:       {price}")
print(f"Description: {description}")

### How `find()` Works

```python
soup.find(tag_name, class_=css_class)
```

- **`tag_name`**: The HTML element to search for (e.g., `"h1"`, `"span"`, `"div"`)
- **`class_`**: Filter by CSS class (note the trailing underscore — `class` is a Python keyword)
- **Returns**: The first matching element as a Beautiful Soup `Tag` object, or `None` if not found
- **`.get_text(strip=True)`**: Extracts just the text content, removing HTML tags and extra whitespace

In an agent pipeline, the parser produces a clean product record; the LLM then generates a summary, compares against competitors, or drafts marketing copy. **Parsing is deterministic; the LLM handles synthesis and reasoning.**

## Example 2: Extracting a Table into Structured Data

HTML tables are extremely common in business contexts — financial reports, inventory lists, schedules. Let's extract a table into a list of dictionaries.

In [ ]:
html_table = """
<html>
<body>
  <h2>Q1 2026 Sales Report</h2>
  <table>
    <thead>
      <tr>
        <th>Product</th>
        <th>Units Sold</th>
        <th>Revenue</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td>Widget A</td>
        <td>1,240</td>
        <td>$24,800.00</td>
      </tr>
      <tr>
        <td>Widget B</td>
        <td>870</td>
        <td>$43,500.00</td>
      </tr>
      <tr>
        <td>Widget C</td>
        <td>2,100</td>
        <td>$10,500.00</td>
      </tr>
    </tbody>
  </table>
</body>
</html>
"""

soup = BeautifulSoup(html_table, "html.parser")

# Extract the report title
report_title = soup.find("h2").get_text(strip=True)
print(f"Report: {report_title}\n")

# Extract column headers
headers = [th.get_text(strip=True) for th in soup.find("thead").find_all("th")]
print(f"Columns: {headers}")

# Extract each row into a dictionary
rows = []
for tr in soup.find("tbody").find_all("tr"):
    cells = [td.get_text(strip=True) for td in tr.find_all("td")]
    row = dict(zip(headers, cells))
    rows.append(row)

# Display the extracted data
for row in rows:
    print(row)

### Key Parsing Methods

| Method | Purpose |
|--------|--------|
| `soup.find(tag)` | Find the **first** element matching the tag |
| `soup.find_all(tag)` | Find **all** elements matching the tag (returns a list) |
| `tag.get_text(strip=True)` | Extract text content from an element |
| `tag["attribute"]` | Access an HTML attribute (e.g., `tag["href"]` for links) |
| `tag.find(tag2)` | Search **within** a specific element |

## Example 3: Extracting Links and Their Text

A research agent crawling the web needs to find and follow links. Beautiful Soup makes it easy to extract both the URL and the anchor text.

In [ ]:
html_links = """
<html>
<body>
  <h1>Useful Resources</h1>
  <ul>
    <li><a href="https://docs.python.org/3/library/re.html">Python regex documentation</a></li>
    <li><a href="https://www.crummy.com/software/BeautifulSoup/">Beautiful Soup homepage</a></li>
    <li><a href="https://developer.mozilla.org/en-US/docs/Web/HTML">MDN HTML reference</a></li>
  </ul>
</body>
</html>
"""

soup = BeautifulSoup(html_links, "html.parser")

# find_all("a") finds all anchor tags
links = soup.find_all("a")

print("Extracted links:")
for link in links:
    url = link["href"]           # access the href attribute
    text = link.get_text(strip=True)  # get the visible text
    print(f"  {text}")
    print(f"    -> {url}\n")

## Example 4: Cleaning HTML — Removing Unwanted Elements

Real web pages are cluttered with navigation, ads, and scripts. Before extracting content, you often need to **strip away the noise**.

In [ ]:
messy_html = """
<html>
<head><title>Article Page</title></head>
<body>
  <nav>
    <a href="/">Home</a> | <a href="/about">About</a> | <a href="/contact">Contact</a>
  </nav>
  
  <div class="ad-banner">
    <p>BUY NOW! Special offer on widgets!</p>
  </div>
  
  <article>
    <h1>Understanding NLP Preprocessing</h1>
    <p>Natural Language Processing begins with converting raw text into a format
    that machines can work with. This involves several key steps including
    tokenization, normalization, and feature extraction.</p>
    <p>In agentic AI systems, preprocessing serves as the bridge between
    unstructured inputs and structured reasoning.</p>
  </article>
  
  <script>
    console.log('tracking pixel loaded');
  </script>
  
  <footer>
    <p>Copyright 2026 Example Corp</p>
  </footer>
</body>
</html>
"""

soup = BeautifulSoup(messy_html, "html.parser")

# Remove unwanted elements before extracting content
for tag in soup.find_all(["nav", "script", "footer"]):
    tag.decompose()  # removes the tag and its contents entirely

for ad in soup.find_all("div", class_="ad-banner"):
    ad.decompose()

# Now extract just the article content
article = soup.find("article")
if article:
    title = article.find("h1").get_text(strip=True)
    paragraphs = [p.get_text(strip=True) for p in article.find_all("p")]
    
    print(f"Title: {title}\n")
    print("Content:")
    for p in paragraphs:
        print(f"  {p}\n")

### Key Concept: `decompose()` vs `extract()`

- **`tag.decompose()`**: Destroys the tag and its contents completely. Use when you want to clean up the HTML before further processing.
- **`tag.extract()`**: Removes the tag from the tree but returns it, so you can still use it. Use when you want to separate parts of the document.

---

# Part 4: Combining Regex and HTML Parsing

In real-world agent pipelines, you often **combine** both techniques: Beautiful Soup extracts text from HTML, then regex extracts structured fields from that text.

In [ ]:
import re
from bs4 import BeautifulSoup

# A product listing page with embedded structured data
product_page = """
<html>
<body>
  <div class="product-listing">
    <div class="product">
      <h3>Ergonomic Keyboard Pro</h3>
      <p class="details">SKU: KB-2026-001 | In stock: 142 units | Price: $89.99</p>
      <p class="rating">Customer rating: 4.7/5 (238 reviews)</p>
    </div>
    <div class="product">
      <h3>Wireless Mouse Ultra</h3>
      <p class="details">SKU: MS-2026-045 | In stock: 67 units | Price: $45.50</p>
      <p class="rating">Customer rating: 4.3/5 (189 reviews)</p>
    </div>
    <div class="product">
      <h3>USB-C Hub Premium</h3>
      <p class="details">SKU: HB-2026-012 | In stock: 0 units | Price: $34.99</p>
      <p class="rating">Customer rating: 4.5/5 (412 reviews)</p>
    </div>
  </div>
</body>
</html>
"""

soup = BeautifulSoup(product_page, "html.parser")

# Step 1: Beautiful Soup extracts each product block
products = []
for product_div in soup.find_all("div", class_="product"):
    name = product_div.find("h3").get_text(strip=True)
    details_text = product_div.find("p", class_="details").get_text(strip=True)
    rating_text = product_div.find("p", class_="rating").get_text(strip=True)
    
    # Step 2: Regex extracts structured fields from the text
    sku = re.search(r"SKU: ([A-Z]{2}-\d{4}-\d{3})", details_text).group(1)
    stock = int(re.search(r"In stock: (\d+)", details_text).group(1))
    price = float(re.search(r"Price: \$(\d+\.\d{2})", details_text).group(1))
    rating = float(re.search(r"(\d\.\d)/5", rating_text).group(1))
    reviews = int(re.search(r"\((\d+) reviews\)", rating_text).group(1))
    
    products.append({
        "name": name,
        "sku": sku,
        "stock": stock,
        "price": price,
        "rating": rating,
        "reviews": reviews
    })

# Display structured results
print(f"{'Product':<25} {'SKU':<15} {'Stock':>5} {'Price':>8} {'Rating':>6} {'Reviews':>7}")
print("-" * 75)
for p in products:
    print(f"{p['name']:<25} {p['sku']:<15} {p['stock']:>5} ${p['price']:>7.2f} {p['rating']:>5.1f} {p['reviews']:>7}")

# Flag out-of-stock items — this structured data could be sent to an LLM
# for generating restock alerts or customer notifications
out_of_stock = [p for p in products if p['stock'] == 0]
if out_of_stock:
    print(f"\n⚠ Out of stock: {', '.join(p['name'] for p in out_of_stock)}")

---

# Part 5: The Big Picture — Where This Fits in Agentic AI

Let's step back and see how these preprocessing techniques fit into a complete agentic workflow:

```
┌─────────────────────────────────────────────────────────────────┐
│                    AGENTIC AI PIPELINE                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. RAW INPUT                                                   │
│     Emails, HTML pages, log files, PDFs, chat transcripts       │
│                          ↓                                      │
│  2. DETERMINISTIC PREPROCESSING  ← ── You are here             │
│     • Beautiful Soup: parse HTML, extract clean text             │
│     • Regex: extract dates, amounts, IDs, patterns              │
│     • Encoding normalization, deduplication                      │
│                          ↓                                      │
│  3. STRUCTURED DATA                                             │
│     Clean text + extracted fields (JSON, dicts, DataFrames)      │
│                          ↓                                      │
│  4. LLM REASONING                                               │
│     • Summarization    • Classification    • Decision-making     │
│     • Tool calling     • Response generation                     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

**The key insight**: Use deterministic methods (regex, parsing) for what they do best (structured extraction), and use LLMs for what *they* do best (reasoning, synthesis, generation). Combining both gives you a system that is reliable, efficient, and capable.

---

# Practice Exercises

Try these exercises to reinforce what you've learned.

## Exercise 1: Extract Phone Numbers

Write a regex to extract all US phone numbers from the text below. Phone numbers may appear as `(555) 123-4567`, `555-123-4567`, or `555.123.4567`.

In [ ]:
contact_info = """
Main office: (555) 234-5678
Sales team: 555-876-5432
Support hotline: 555.111.2222
Fax (do people still fax?): (555) 999-0000
"""

# TODO: Write a regex pattern that matches all four phone number formats
phone_pattern = r""  # Your pattern here

phones = re.findall(phone_pattern, contact_info)
print("Phone numbers found:", phones)

## Exercise 2: Parse a Job Listings Page

Extract job title, company, and salary from the HTML below.

In [ ]:
jobs_html = """
<html>
<body>
  <div class="job-listing">
    <h2 class="job-title">Data Scientist</h2>
    <span class="company">Acme Analytics</span>
    <span class="salary">$95,000 - $120,000</span>
  </div>
  <div class="job-listing">
    <h2 class="job-title">ML Engineer</h2>
    <span class="company">TechCorp AI</span>
    <span class="salary">$110,000 - $145,000</span>
  </div>
  <div class="job-listing">
    <h2 class="job-title">NLP Researcher</h2>
    <span class="company">Language Labs</span>
    <span class="salary">$100,000 - $135,000</span>
  </div>
</body>
</html>
"""

# TODO: Use Beautiful Soup to extract each job listing into a dictionary
# with keys: "title", "company", "salary"

# Your code here

## Exercise 3: Build a Mini Pipeline

Combine regex and Beautiful Soup: parse the HTML to extract the text, then use regex to pull out specific data points.

In [ ]:
report_html = """
<html>
<body>
  <article class="report">
    <h1>Monthly Status Report</h1>
    <p>Report generated on 2026-03-01 by the analytics team.</p>
    <p>Key metrics: Revenue was $4,523,100.00 this month, up from
    $4,102,750.00 last month. Customer count reached 12,847.</p>
    <p>Contact: reports@analytics-team.com for questions.</p>
  </article>
  <nav><a href="/">Home</a></nav>
</body>
</html>
"""

# TODO: 
# 1. Use Beautiful Soup to extract just the article text (ignore nav)
# 2. Use regex to extract: the date, all dollar amounts, the email address
# 3. Print the extracted structured data

# Your code here

---

## Summary

| Technique | Best For | Key Functions |
|-----------|----------|---------------|
| **Regular Expressions** | Pattern matching in semi-structured text (dates, IDs, amounts, emails) | `re.findall()`, `re.search()`, `re.sub()`, `re.finditer()` |
| **Beautiful Soup** | Navigating and extracting content from HTML/XML documents | `find()`, `find_all()`, `get_text()`, `decompose()` |
| **Combined pipeline** | Real-world agent preprocessing — parse HTML first, then regex for structured fields | Both libraries together |

### Key Takeaways

1. **Deterministic preprocessing** (regex, HTML parsing) is the critical first step in any agentic AI pipeline
2. **Regex** is fast, transparent, and reliable for extracting known patterns from text
3. **Beautiful Soup** handles the messy reality of HTML — finding elements by tag, class, and attribute
4. **Combine both** in real pipelines: parse HTML to get clean text, then use regex to extract structured fields
5. **Reserve LLMs** for what they do best: reasoning, synthesis, and generation — not deterministic extraction